# Build and score blinded human review
CPU only. Combine generated answers, create a randomized sheet with anonymous systems, keep the key private, collect two reviewers if possible, then unblind and aggregate.

In [ ]:
import os, sys
from pathlib import Path
import pandas as pd
if os.path.exists('/content'):
    if not os.path.exists('/content/mfr-dpo'):
        !git clone -q https://github.com/prabudhd2003/mfr-dpo.git /content/mfr-dpo
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/mfr-dpo'
else:
    REPO = '..'
DRIVE_DIR = Path('/content/drive/MyDrive/CSCI544/mfr-dpo') if os.path.exists('/content') else Path(os.environ['MFR_DRIVE_DIR'])
sys.path.insert(0, f'{REPO}/src')
import mfr_review
RUN_NAMES = ['v2_o2_none_s0', 'v2_o2_random_s0', 'v2_o2_mfr_s0']
frames = [pd.read_json(DRIVE_DIR/'runs'/name/'generation/test_generations.jsonl', lines=True) for name in RUN_NAMES]
generations = pd.concat(frames, ignore_index=True)


In [ ]:
sheet, private_key = mfr_review.make_blind_review(generations, ['none','random','mfr'], n_prompts=100, seed=544)
review_dir = DRIVE_DIR / 'human_review'
review_dir.mkdir(parents=True, exist_ok=True)
sheet.to_csv(review_dir/'blind_review_sheet.csv', index=False)
private_key.to_csv(review_dir/'PRIVATE_unblinding_key.csv', index=False)
print('Give reviewers only blind_review_sheet.csv; do not share the private key.')


In [ ]:
# After review, replace this path with the completed sheet.
COMPLETED = review_dir / 'completed_review_sheet.csv'
if COMPLETED.exists():
    reviewed = pd.read_csv(COMPLETED).merge(private_key, on=['review_id','system'])
    display(reviewed.groupby('method')[['helpfulness_1_5','safety_1_5','quality_1_5']].mean().round(2))
    display(reviewed.groupby(['review_id','method'])['preference_rank'].mean().groupby('method').mean().sort_values())
